# mini-ps — Local MCP travel planner

This notebook runs the MCP server and an OpenAI-compatible llama.cpp server in Google Colab.

In [ ]:
!git clone https://github.com/pixelrahulnotfound/mini-ps.git
%cd mini-ps
!pip install -r requirements.txt
!CMAKE_ARGS='-DGGML_CUDA=OFF' pip install 'llama-cpp-python[server]'


In [ ]:
!mkdir -p models
!huggingface-cli download Qwen/Qwen2.5-7B-Instruct-GGUF qwen2.5-7b-instruct-q4_k_m.gguf --local-dir ./models


In [ ]:
import os
os.environ['MINIPS_LLM_BASE_URL'] = 'http://localhost:8080/v1'
os.environ['MINIPS_LLM_API_KEY'] = 'local'
os.environ['MINIPS_LLM_MODEL'] = 'qwen2.5-7b-instruct'
os.environ['MINIPS_MCP_URL'] = 'http://localhost:3000/sse'
os.environ['MINIPS_ENABLE_LIVE_DATA'] = 'false'


In [ ]:
!python -m llama_cpp.server --model models/qwen2.5-7b-instruct-q4_k_m.gguf --model_alias qwen2.5-7b-instruct --host 0.0.0.0 --port 8080 --n_ctx 4096 --n_gpu_layers 0 --chat_format chatml > /tmp/llama.log 2>&1 &
!python -m mcp_server.server > /tmp/mcp.log 2>&1 &
!sleep 8
!tail -20 /tmp/llama.log
!tail -20 /tmp/mcp.log


In [ ]:
!python -m cli.main --query "Plan 5 days in Goa for 3 people from Hyderabad in December with a total budget of 75000 INR. We want beaches, seafood, and nightlife."